# H2a / H2b — Populism and accusations of lying

**H2a**  populist politicians are more likely to *accuse* opponents of lying.
**H2b**  populist politicians are more likely to *be accused* of lying.

### Design

Unit of analysis is the **speaker-year** (`lib/panel.py`): one row per MP per year
with `n_sentences` spoken (the exposure), accusations made, accusations received,
and all individual- and party-level covariates.

Count models with `offset(log n_sentences)`, so every coefficient is about the
*rate per sentence spoken* — an MP who accuses a lot simply because they talk a
lot is not counted as more accusatory.

**Poisson with cluster-robust SEs is the primary specification.** Poisson QMLE is
consistent for the conditional mean even when the data are overdispersed (which
they badly are — see H1), whereas the negative binomial is only consistent if its
variance function is right. The NB is reported alongside as a check.

Populism = V-Party `v2xpa_popul` for the MP's party in that year. Because
populism turned out to be nearly **uncorrelated with cultural conservatism**
(r ≈ 0.05) and only modestly related to economic left-right (r ≈ -0.35), H2 and
H4 can be estimated in one model without the variance inflation we feared.

### The asymmetry between the two sides

H2a is clean. **H2b is not**: `n_accusations_received` only counts accusations
whose target could be *resolved* to a known speaker (~50% overall, and varying
strongly by parliament — Westminster-style indirect reference resolves worst).
Every accusee model therefore carries source-dataset fixed effects, and section 5
re-runs it on high-resolution datasets only.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

from lib import data, panel, viz
from lib.codebooks import EXCLUDED_COUNTRIES
viz.apply_style()

MIN_SENTENCES = 50          # speaker-years below this have too little exposure
YEAR_MIN, YEAR_MAX = 1994, 2022    # same window as H1

print(f"excluded countries: {sorted(EXCLUDED_COUNTRIES)}")
print(f"window: {YEAR_MIN}-{YEAR_MAX}")

## 1. Estimation sample

The panel is cached. If the excluded-country list changed since it was built,
rebuild it first:

```bash
cd ~/projects/corpora/analysis && python3 -m lib.panel --force
```

In [ ]:
df = panel.load_panel()
print(f"panel: {len(df):,} speaker-years, {df['speaker_id'].nunique():,} speakers")

est = df[(df["speaker_match"] == "resolved")
         & df["year"].between(YEAR_MIN, YEAR_MAX)
         & (df["n_sentences"] >= MIN_SENTENCES)].copy()
est = est.dropna(subset=["populism", "left_right", "cultural_conservatism",
                         "female", "age"])
est["edu"] = est["highest_isced"]

print(f"\nestimation sample: {len(est):,} speaker-years")
print(f"  speakers  : {est['speaker_id'].nunique():,}")
print(f"  countries : {est['country'].nunique()}")
print(f"  years     : {est['year'].min()}-{est['year'].max()}")
print(f"  sentences : {est['n_sentences'].sum():,}")
print(f"\naccusations made     : {est['n_accusations_made'].sum():,}")
print(f"accusations received : {est['n_accusations_received'].sum():,}")
print(f"  (received is resolved-target only -- see the design note)")

### Are the regressors separable?

If populism, economic left-right and cultural conservatism were strongly
collinear, a joint model could not distinguish H2 from H4. This is the check.

In [ ]:
REGRESSORS = ["populism", "left_right", "cultural_conservatism",
              "anti_pluralism", "female", "age", "edu", "in_cabinet",
              "vote_share_last"]

print("correlations among regressors:\n")
print(est[REGRESSORS].corr().round(2).to_string())
print("\ndescriptives:\n")
print(est[REGRESSORS].describe().T[["count", "mean", "std", "min", "max"]]
        .round(2).to_string())

## 2. Descriptive — rates by populism quartile

Raw rates, no controls. If H2a and H2b hold, both bars should rise across
quartiles.

In [ ]:
est["pop_q"] = pd.qcut(est["populism"], 4,
                       labels=["Q1 least", "Q2", "Q3", "Q4 most populist"])

desc = est.groupby("pop_q", observed=True).apply(
    lambda g: pd.Series({
        "made_per_10k": g["n_accusations_made"].sum() / g["n_sentences"].sum() * 1e4,
        "received_per_10k": g["n_accusations_received"].sum() / g["n_sentences"].sum() * 1e4,
        "speaker_years": len(g),
    }), include_groups=False)

print(desc.round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(desc)); w = 0.38
ax.bar(x - w/2, desc["made_per_10k"], w, label="accusations made")
ax.bar(x + w/2, desc["received_per_10k"], w, label="accusations received")
ax.set_xticks(x); ax.set_xticklabels(desc.index, fontsize=9)
ax.set_ylabel("per 10,000 sentences spoken")
ax.set_title("Accusations made and received, by party-populism quartile")
ax.legend()
viz.savefig(fig, "h2_descriptive_quartiles")
plt.show()

## 3. Models

One specification per side. `report()` prints the incidence-rate ratio (IRR):
the multiplicative effect on the accusation rate. IRR > 1 means more accusations.

Populism runs 0–1, so its IRR is the effect of moving from the least to the most
populist party in the data — a large, interpretable contrast.

In [ ]:
CONTROLS = ("left_right + cultural_conservatism + female + age + edu "
            "+ in_cabinet + vote_share_last")

FOCAL = ["populism", "left_right", "cultural_conservatism", "female", "age",
         "edu", "in_cabinet"]


def fit_count(dv, rhs, d, family="poisson", cluster="speaker_id"):
    fam = (sm.families.Poisson() if family == "poisson"
           else sm.families.NegativeBinomial(alpha=1.0))
    return (smf.glm(f"{dv} ~ {rhs}", data=d, family=fam,
                    offset=d["log_exposure"])
               .fit(cov_type="cluster", cov_kwds={"groups": d[cluster]}))


def report(m, label, terms=FOCAL):
    print(f"\n--- {label}  (n={int(m.nobs):,}) ---")
    print(f"{'term':<24}{'IRR':>8}{'95% CI':>22}{'p':>10}")
    for t in terms:
        if t not in m.params.index:
            continue
        b = m.params[t]; lo, hi = m.conf_int().loc[t]
        print(f"{t:<24}{np.exp(b):>8.3f}"
              f"{f'[{np.exp(lo):.3f}, {np.exp(hi):.3f}]':>22}"
              f"{m.pvalues[t]:>10.3g}")
    return m

### 3a. H2a — accuser side

DV = accusations **made**. Country and year fixed effects: we are asking whether,
*within a parliament and year*, MPs from more populist parties accuse more.

In [ ]:
m_made = fit_count("n_accusations_made",
                   f"populism + {CONTROLS} + C(country) + C(year)", est)
report(m_made, "H2a: accusations MADE (Poisson, country+year FE)")

irr = np.exp(m_made.params["populism"])
print(f"\nH2a expects IRR > 1 for populism. Observed: {irr:.2f}")
print(f"=> MPs of a maximally populist party accuse "
      f"{(irr - 1) * 100:+.0f}% more often per sentence spoken")

### 3b. H2b — accusee side

DV = accusations **received**. `source_dataset` fixed effects instead of country,
because target resolution varies by corpus and that is the main threat here.

In [ ]:
m_recv = fit_count("n_accusations_received",
                   f"populism + {CONTROLS} + C(source_dataset) + C(year)", est)
report(m_recv, "H2b: accusations RECEIVED (Poisson, dataset+year FE)")

irr = np.exp(m_recv.params["populism"])
print(f"\nH2b expects IRR > 1 for populism. Observed: {irr:.2f}")
print(f"=> MPs of a maximally populist party are accused "
      f"{(irr - 1) * 100:+.0f}% more often per sentence spoken")

### 3c. Build-up — does populism survive controls?

Bivariate, then + individual controls, then the full model. If the populism
coefficient collapses once ideology enters, the populism story is really an
ideology story.

In [ ]:
steps = {
    "bivariate":        "populism + C(country) + C(year)",
    "+ individual":     "populism + female + age + edu + C(country) + C(year)",
    "+ ideology":       ("populism + left_right + cultural_conservatism + female "
                         "+ age + edu + C(country) + C(year)"),
    "full":             f"populism + {CONTROLS} + C(country) + C(year)",
}

rows = []
for label, rhs in steps.items():
    m = fit_count("n_accusations_made", rhs, est)
    b = m.params["populism"]; lo, hi = m.conf_int().loc["populism"]
    rows.append({"model": label, "IRR": np.exp(b),
                 "lo": np.exp(lo), "hi": np.exp(hi),
                 "p": m.pvalues["populism"]})
build = pd.DataFrame(rows)
print("H2a populism coefficient across specifications:\n")
print(build.round(3).to_string(index=False))

## 4. Coefficient plot

Both sides together. Points right of 1 mean more accusations.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for i, (m, lab, col) in enumerate([(m_made, "made (H2a)", "#c0603f"),
                                   (m_recv, "received (H2b)", "#2f6f9f")]):
    y = np.arange(len(FOCAL)) + (i - 0.5) * 0.28
    for j, t in enumerate(FOCAL):
        if t not in m.params.index:
            continue
        b = m.params[t]; lo, hi = m.conf_int().loc[t]
        ax.plot([np.exp(lo), np.exp(hi)], [y[j], y[j]], color=col, lw=2)
        ax.plot(np.exp(b), y[j], "o", color=col, ms=5,
                label=lab if j == 0 else None)

ax.axvline(1, color="black", lw=1, zorder=0)
ax.set_yticks(np.arange(len(FOCAL))); ax.set_yticklabels(FOCAL)
ax.set_xscale("log")
ax.set_xlabel("incidence-rate ratio (log scale, 95% CI)")
ax.set_title("Drivers of accusations of lying — speaker-year models")
ax.legend(loc="lower right")
fig.tight_layout()
viz.savefig(fig, "h2_coefplot")
plt.show()

## 5. Robustness

### 5a. Negative binomial

Poisson QMLE is consistent under overdispersion; NB is the conventional
alternative. Materially different results would be a warning.

In [ ]:
nb_made = fit_count("n_accusations_made",
                    f"populism + {CONTROLS} + C(country) + C(year)", est,
                    family="nb")
nb_recv = fit_count("n_accusations_received",
                    f"populism + {CONTROLS} + C(source_dataset) + C(year)", est,
                    family="nb")

comp = pd.DataFrame({
    "Poisson made":  [np.exp(m_made.params["populism"])],
    "NB made":       [np.exp(nb_made.params["populism"])],
    "Poisson recv":  [np.exp(m_recv.params["populism"])],
    "NB recv":       [np.exp(nb_recv.params["populism"])],
}, index=["populism IRR"])
print(comp.round(3).to_string())

### 5b. Alternative populism measures

`v2xpa_popul` is an index of anti-elitism and people-centrism. Running its two
components separately, plus anti-pluralism, shows whether the result is driven by
one facet.

In [ ]:
alt_rows = []
for var in ["populism", "anti_elitism", "people_centrism", "anti_pluralism"]:
    d = est.dropna(subset=[var])
    if d[var].nunique() < 10:
        print(f"({var}: insufficient variation, skipped)")
        continue
    for dv, fe, side in [("n_accusations_made", "C(country)", "made"),
                         ("n_accusations_received", "C(source_dataset)", "received")]:
        m = fit_count(dv, f"{var} + {CONTROLS} + {fe} + C(year)", d)
        b = m.params[var]; lo, hi = m.conf_int().loc[var]
        alt_rows.append({"measure": var, "side": side, "IRR": np.exp(b),
                         "lo": np.exp(lo), "hi": np.exp(hi), "p": m.pvalues[var],
                         "n": int(m.nobs)})
alt = pd.DataFrame(alt_rows)
print(alt.round(3).to_string(index=False))

### 5c. Binary populist indicator

A continuous index assumes a linear effect. A simple "is this a populist party"
dummy is closer to how the literature actually talks.

In [ ]:
est["populist_party"] = (est["populism"] > 0.5).astype(int)
print(f"populist party-years: {int(est['populist_party'].sum()):,} "
      f"of {len(est):,} ({est['populist_party'].mean()*100:.1f}%)\n")

for dv, fe, side in [("n_accusations_made", "C(country)", "made (H2a)"),
                     ("n_accusations_received", "C(source_dataset)", "received (H2b)")]:
    m = fit_count(dv, f"populist_party + {CONTROLS} + {fe} + C(year)", est)
    b = m.params["populist_party"]; lo, hi = m.conf_int().loc["populist_party"]
    print(f"{side:<18} IRR = {np.exp(b):.3f}  "
          f"[{np.exp(lo):.3f}, {np.exp(hi):.3f}]  p = {m.pvalues['populist_party']:.3g}")

### 5d. H2b on high-resolution datasets only

The accusee results depend on targets being resolvable. This restricts to corpora
where a decent share of person-targets were resolved, so the DV is less censored.

In [ ]:
con = data.duck()
res = con.execute("""
    SELECT source_dataset,
           COUNT(*) AS n_person_targets,
           SUM(CASE WHEN target_speaker_id IS NOT NULL THEN 1 ELSE 0 END) AS n_resolved
    FROM accusations
    WHERE target_type = 'person'
    GROUP BY 1
    ORDER BY 1
""").df()
res["resolution_rate"] = res["n_resolved"] / res["n_person_targets"]
print(res.round(3).to_string(index=False))

MIN_RES = 0.40
good = set(res.loc[res["resolution_rate"] >= MIN_RES, "source_dataset"])
sub = est[est["source_dataset"].isin(good)].copy()
print(f"\ndatasets with resolution >= {MIN_RES:.0%}: {len(good)} "
      f"-> {len(sub):,} speaker-years")

if len(sub) > 1000 and sub["source_dataset"].nunique() >= 2:
    m = fit_count("n_accusations_received",
                  f"populism + {CONTROLS} + C(source_dataset) + C(year)", sub)
    report(m, f"H2b, resolution >= {MIN_RES:.0%}")
else:
    print("too few observations for a restricted model")

### 5e. Does populism interact with ideology?

The literature emphasises the *radical right* specifically. If populism only
raises accusations among culturally conservative parties, the interaction is
positive and the populism main effect is a left-populist story too.

In [ ]:
m_int = fit_count("n_accusations_made",
                  f"populism * cultural_conservatism + {CONTROLS.replace('cultural_conservatism + ', '')}"
                  f" + C(country) + C(year)", est)

terms = [t for t in m_int.params.index
         if "populism" in t or "cultural_conservatism" in t]
print("H2a with populism x cultural conservatism:\n")
for t in terms:
    b = m_int.params[t]; lo, hi = m_int.conf_int().loc[t]
    print(f"  {t:<42} IRR {np.exp(b):>7.3f}  "
          f"[{np.exp(lo):.3f}, {np.exp(hi):.3f}]  p={m_int.pvalues[t]:.3g}")
print("\nPositive interaction => populism matters more on the cultural right.")

## 6. Verdict

| | populism IRR | 95% CI | p | verdict |
|---|---|---|---|---|
| H2a accusations made | | | | |
| H2b accusations received | | | | |

**H2a supported** if the made-side IRR is above 1 and survives section 3c's
build-up — i.e. populism still predicts accusing after ideology is controlled.

**H2b supported** if the received-side IRR is above 1 *and* holds in 5d. If it
appears only in the full sample and vanishes on high-resolution corpora, the
result is about who gets *named* rather than who gets *accused*.

Three things to watch:

- if populism survives while `cultural_conservatism` and `left_right` are null,
  the populism mechanism is separable from the right-wing one — a sharper claim
  than either H2 or H4 makes alone, and worth foregrounding;
- if 5b shows anti-elitism carrying the effect while people-centrism does not,
  the mechanism is the *elite-attacking* facet, which fits the accusation act
  directly;
- a positive interaction in 5e means the effect concentrates in the radical
  right, which is what Hameleers and Törnberg & Chueri would predict.